# CineGraph RAG — Lab Exercise 3
**Advance NLP (AML23702) - Laasya S D(1GA23AI021)**

To design and implement a small RAG application that uses
the Neo4j Movies dataset as a knowledge graph backend,
retrieves relevant movie/actor/director information via
Cypher queries, and uses an LLM to generate natural
language answers to user questions.


## 1. Install dependencies

In [ ]:
%pip -q install neo4j google-genai streamlit pandas

## 2. Configure credentials
Added `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, and `GEMINI_API_KEY` in Colab Secrets and enable notebook access. Missing values are prompted securely.

In [ ]:
import os, getpass
def setting(name, prompt, secret=True):
    value=os.environ.get(name, "").strip()
    try:
        from google.colab import userdata
        if not value: value=(userdata.get(name) or "").strip()
    except Exception: pass
    if not value: value=getpass.getpass(prompt) if secret else input(prompt)
    if not value: raise ValueError("Missing setting: "+name)
    return value
NEO4J_URI=setting("NEO4J_URI","Neo4j AuraDB URI: ",False)
NEO4J_USERNAME=setting("NEO4J_USERNAME","Neo4j username: ",False)
NEO4J_PASSWORD=setting("NEO4J_PASSWORD","Neo4j password: ")
GEMINI_API_KEY=setting("GEMINI_API_KEY","Gemini API key: ")
print("Credentials loaded; values hidden.")

## 3. Connect and inspect graph schema

In [ ]:

from neo4j import GraphDatabase
import pandas as pd

# Close an existing driver safely, if one exists
try:
    driver.close()
except (NameError, Exception):
    pass

# Connect to your Neo4j AuraDB instance
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

# Verify that the database connection is active
driver.verify_connectivity()
print("✅ Neo4j connection established successfully!")

# Helper function to execute Cypher queries
def run_cypher(query, params=None):
    with driver.session() as session:
        result = session.run(query, params or {})
        return [record.data() for record in result]

# Inspect node labels
labels = run_cypher("""
    CALL db.labels()
    YIELD label
    RETURN label
    ORDER BY label
""")
print("Node labels:", [r["label"] for r in labels])

# Inspect relationship types
relationships = run_cypher("""
    CALL db.relationshipTypes()
    YIELD relationshipType
    RETURN relationshipType
    ORDER BY relationshipType
""")
print("Relationship types:", [
    r["relationshipType"] for r in relationships
])

# Inspect node properties
schema = run_cypher("""
    MATCH (n)
    RETURN labels(n) AS labels,
           keys(n) AS properties
    LIMIT 8
""")

display(pd.DataFrame(schema))

# Check graph size
counts = run_cypher("""
    MATCH (n)
    WITH count(n) AS nodes
    MATCH ()-[r]->()
    RETURN nodes, count(r) AS relationships
""")

display(pd.DataFrame(counts))


## 4. Load Movies sample if needed
If your instance is empty, follow Neo4j's official example-data guide to load the Movies sample, then rerun Section 3. The implementation uses the required Movies dataset rather than substituting another corpus.
- https://neo4j.com/developer/example-data/
- https://neo4j.com/docs/cypher-manual/current/

## 5. Parameterized Cypher query templates

In [ ]:

# SECTION 5: CYPHER QUERY TEMPLATES

QUERIES = {
    "movie": """
        MATCH (m:Movie)
        WHERE toLower(m.title) CONTAINS toLower($term)
        RETURN m.title AS title,
               m.released AS year,
               m.tagline AS tagline
        ORDER BY m.title
        LIMIT 10
    """,

    "cast": """
        MATCH (p:Person)-[r:ACTED_IN]->(m:Movie)
        WHERE toLower(m.title) CONTAINS toLower($term)
        RETURN m.title AS movie,
               m.released AS year,
               p.name AS actor,
               r.roles AS roles
        ORDER BY p.name
        LIMIT 30
    """,

    "actor": """
        MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
        WHERE toLower(p.name) CONTAINS toLower($term)
        RETURN p.name AS actor,
               m.title AS movie,
               m.released AS year
        ORDER BY m.released DESC, m.title
        LIMIT 30
    """,

    "role": """
        MATCH (p:Person)-[r:ACTED_IN]->(m:Movie)
        WHERE any(
            role IN coalesce(r.roles, [])
            WHERE toLower(role) CONTAINS toLower($term)
        )
        RETURN p.name AS actor,
               m.title AS movie,
               m.released AS year,
               r.roles AS roles
        ORDER BY m.title
        LIMIT 20
    """,

    "director": """
        MATCH (p:Person)-[:DIRECTED]->(m:Movie)
        WHERE toLower(m.title) CONTAINS toLower($term)
        RETURN p.name AS director,
               m.title AS movie,
               m.released AS year
        ORDER BY p.name
        LIMIT 20
    """,

    "recommend": """
        MATCH (m:Movie)
        WHERE toLower(m.title) CONTAINS toLower($term)
        MATCH (m)<-[:ACTED_IN]-(p:Person)-[:ACTED_IN]->(other:Movie)
        WHERE other <> m
        WITH other, collect(DISTINCT p.name) AS shared_people
        RETURN other.title AS title,
               other.released AS year,
               shared_people,
               size(shared_people) AS connection_count
        ORDER BY connection_count DESC, year DESC
        LIMIT 8
    """
}

print("✅ Query templates loaded:")
for query_name in QUERIES:
    print("-", query_name)


## 6. Natural-language question routing and retrieval

In [ ]:

# SECTION 6: QUESTION ROUTING AND GRAPH RETRIEVAL

import re

def route_question(question):
    q = question.strip()
    low = q.lower()

    patterns = []

    if any(word in low for word in [
        "recommend", "similar to", "movies like", "films like"
    ]):
        intent = "recommend"
        patterns = [
            r"recommend movies like (.+)",
            r"recommend films like (.+)",
            r"movies like (.+)",
            r"films like (.+)",
            r"similar to (.+)"
        ]

    elif any(word in low for word in [
        "who directed", "director of", "who is the director"
    ]):
        intent = "director"
        patterns = [
            r"who directed (.+)",
            r"director of (.+)",
            r"who is the director of (.+)"
        ]

    elif any(word in low for word in [
        "who played", "who voiced", "who portrayed"
    ]):
        intent = "role"
        patterns = [
            r"who played (.+)",
            r"who voiced (.+)",
            r"who portrayed (.+)"
        ]

    elif any(word in low for word in [
        "who acted in", "cast of", "actors in", "who is in"
    ]):
        intent = "cast"
        patterns = [
            r"who acted in (.+)",
            r"cast of (.+)",
            r"actors in (.+)",
            r"who is in (.+)"
        ]

    elif any(word in low for word in [
        "movies did", "films did", "filmography", "acted in"
    ]):
        intent = "actor"
        patterns = [
            r"what movies did (.+?) act in",
            r"which movies did (.+?) act in",
            r"what films did (.+?) act in",
            r"films did (.+?) act in",
            r"filmography of (.+)"
        ]

    else:
        intent = "movie"
        patterns = [
            r"tell me about (.+)",
            r"details (?:of|about) (.+)",
            r"information (?:on|about) (.+)",
            r"what is (.+)",
            r"show me (.+)"
        ]

    term = q

    for pattern in patterns:
        match = re.search(pattern, low)

        if match:
            term = match.group(1).strip(" ?.!\"'")
            break

    return intent, term.strip(" .!?")


def retrieve(question):
    intent, term = route_question(question)

    records = run_cypher(
        QUERIES[intent],
        {"term": term}
    )

    return {
        "intent": intent,
        "term": term,
        "records": records
    }


print("✅ Question routing and retrieval ready.")

# Quick routing checks
for question in [
    "Tell me about Toy Story",
    "Who acted in Toy Story?",
    "Who directed The Matrix?",
    "What movies did Tom Hanks act in?",
    "Who voiced Woody?",
    "Recommend movies like Heat"
]:
    print(question, "->", route_question(question))


## 7. Gemini grounded answer generation

In [ ]:

# SECTION 7: GROUNDED ANSWER GENERATION

from google import genai

gemini = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL = "gemini-2.5-flash"


def generate_answer(question, result):
    records = result["records"]

    if not records:
        return (
            "I couldn't find matching evidence in your current Neo4j "
            "Movies graph. Try another movie title, actor name, "
            "director, or character role."
        )

    prompt = f"""
You are CineGraph, a movie knowledge-graph assistant.

Answer the user's question using ONLY the Neo4j evidence provided below.

Rules:
1. Do not invent movie titles, actors, directors, roles, or dates.
2. If information is missing from the evidence, say so.
3. Present multiple results as a clear, readable list.
4. For recommendations, explain that they are based on shared actors.
   Do not claim plot or genre similarity unless the evidence supports it.
5. Keep the answer concise and useful.

User question:
{question}

Query intent:
{result["intent"]}

Retrieved Neo4j evidence:
{records}
"""

    try:
        response = gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt
        )

        answer = (response.text or "").strip()

        if answer:
            return answer

        return "The language model returned an empty response. Please try again."

    except Exception as error:
        print("Gemini error:", type(error).__name__)

        return (
            "The graph query retrieved the following evidence, but "
            "Gemini could not generate a response:\n"
            + str(records)
        )


def ask_cinegraph(question):
    result = retrieve(question)
    answer = generate_answer(question, result)

    return answer, result


print("✅ Gemini answer generation ready.")

# Test one question
answer, result = ask_cinegraph("Tell me about Toy Story")

print("\nAnswer:")
print(answer)

print("\nRetrieved evidence:")
display(pd.DataFrame(result["records"]))


## 8. Run a small evaluation set

In [ ]:

# SECTION 8: EVALUATION

tests = [
    "Tell me about Toy Story",
    "Who acted in Toy Story?",
    "Who directed The Matrix?",
    "What movies did Tom Hanks act in?",
    "Who voiced Woody?",
    "Recommend movies like Heat",
    "Tell me about a movie that does not exist"
]

rows = []

for question in tests:
    try:
        answer, result = ask_cinegraph(question)

        rows.append({
            "question": question,
            "intent": result["intent"],
            "search_term": result["term"],
            "evidence_count": len(result["records"]),
            "status": (
                "Evidence found"
                if result["records"]
                else "No matching evidence"
            ),
            "answer_preview": answer[:250]
        })

    except Exception as error:
        rows.append({
            "question": question,
            "intent": "ERROR",
            "search_term": "",
            "evidence_count": 0,
            "status": "Failed",
            "answer_preview": str(error)[:250]
        })

evaluation_df = pd.DataFrame(rows)

display(evaluation_df)

print("\nEvaluation summary:")
print("Total questions:", len(evaluation_df))
print(
    "Questions with graph evidence:",
    int((evaluation_df["evidence_count"] > 0).sum())
)
print(
    "Failed queries:",
    int((evaluation_df["status"] == "Failed").sum())
)


## 9. Create the Streamlit frontend
This creates a separate `/content/app.py` frontend with question suggestions, a dark CineGraph theme, answer output, retrieved evidence, and CSV export.

In [ ]:
app_code = 'import os,re,streamlit as st,pandas as pd\nfrom neo4j import GraphDatabase\nfrom google import genai\nst.set_page_config(page_title="CineGraph",page_icon="🎬",layout="wide")\ndriver=GraphDatabase.driver(os.environ["NEO4J_URI"],auth=(os.environ["NEO4J_USERNAME"],os.environ["NEO4J_PASSWORD"]))\ngemini=genai.Client(api_key=os.environ["GEMINI_API_KEY"]); MODEL="gemini-2.5-flash"\nQUERIES={\'movie\': \'MATCH (m:Movie) WHERE toLower(m.title) CONTAINS toLower($term) RETURN m.title AS title,m.year AS year,m.genres AS genres,m.languages AS languages LIMIT 10\', \'cast\': \'MATCH (p:Person)-[r:ACTED_IN]->(m:Movie) WHERE toLower(m.title) CONTAINS toLower($term) RETURN m.title AS movie,m.year AS year,p.name AS actor,r.roles AS roles,r.role AS role LIMIT 30\', \'actor\': \'MATCH (p:Person)-[r:ACTED_IN]->(m:Movie) WHERE toLower(p.name) CONTAINS toLower($term) RETURN p.name AS actor,m.title AS movie,m.year AS year,r.roles AS roles,r.role AS role LIMIT 30\', \'role\': "MATCH (p:Person)-[r:ACTED_IN]->(m:Movie) WHERE any(x IN coalesce(r.roles,[]) WHERE toLower(x) CONTAINS toLower($term)) OR toLower(coalesce(r.role,\'\')) CONTAINS toLower($term) RETURN p.name AS actor,m.title AS movie,m.year AS year,r.roles AS roles,r.role AS role LIMIT 20", \'recommend\': \'MATCH (m:Movie) WHERE toLower(m.title) CONTAINS toLower($term) MATCH (m)<-[:ACTED_IN]-(p:Person)-[:ACTED_IN]->(o:Movie) WHERE o<>m RETURN o.title AS title,o.year AS year,collect(DISTINCT p.name) AS shared_people,count(DISTINCT p) AS connection_count ORDER BY connection_count DESC LIMIT 8\'}\ndef route(q):\n low=q.lower().strip()\n if any(x in low for x in ["recommend","similar to","movies like","films like"]): intent,pats="recommend",[r"recommend movies like (.+)",r"movies like (.+)",r"similar to (.+)"]\n elif any(x in low for x in ["who played","who voiced","which actor played","who portrayed"]): intent,pats="role",[r"who played (.+)",r"who voiced (.+)",r"which actor played (.+)",r"who portrayed (.+)"]\n elif any(x in low for x in ["who acted in","cast of","actors in","who is in"]): intent,pats="cast",[r"who acted in (.+)",r"cast of (.+)",r"actors in (.+)",r"who is in (.+)"]\n elif any(x in low for x in ["movies did","films did","filmography","acted in"]): intent,pats="actor",[r"what movies did (.+?) act in",r"which movies did (.+?) act in",r"films did (.+?) act in",r"filmography of (.+)"]\n else: intent,pats="movie",[r"tell me about (.+)",r"details (?:of|about) (.+)",r"information (?:on|about) (.+)"]\n term=q\n for pat in pats:\n  m=re.search(pat,low)\n  if m: term=m.group(1).strip(" ?.!\\"\'"); break\n return intent,term.strip(" .!?")\ndef ask(q):\n intent,term=route(q)\n with driver.session() as s: records=[r.data() for r in s.run(QUERIES[intent],{"term":term})]\n if not records: return "No matching graph evidence. Try a more specific title, person, or role.",intent,term,records\n prompt="Answer only from this Neo4j evidence. Do not invent facts. Recommendations use shared actors, not guaranteed plot similarity.\\nQuestion: "+q+"\\nEvidence: "+str(records)\n try: answer=(gemini.models.generate_content(model=MODEL,contents=prompt).text or "").strip()\n except Exception as e: answer="LLM unavailable ("+type(e).__name__+"); inspect retrieved evidence below."\n return answer,intent,term,records\nst.markdown("<div style=\'padding:24px;border-radius:16px;background:linear-gradient(120deg,#182640,#251b3d)\'><small>GRAPH RETRIEVAL · EVIDENCE FIRST</small><h1>🎬 CineGraph</h1><p>Explore the connections behind your favourite films.</p></div>",unsafe_allow_html=True)\nexamples=["Tell me about Toy Story","Who acted in Toy Story?","What movies did Tom Hanks act in?","Who voiced Woody?","Recommend movies like Heat"]\nwith st.sidebar: chosen=st.selectbox("Try a question",["Choose an example…"]+examples); st.caption("Data source: connected Neo4j Movies graph")\nq=st.text_input("Ask CineGraph",value="" if chosen=="Choose an example…" else chosen,placeholder="Who acted in Toy Story?")\nif st.button("Find answer →",type="primary",use_container_width=True):\n if not q.strip(): st.warning("Enter a question first.")\n else:\n  with st.spinner("Querying graph and preparing answer…"): answer,intent,term,records=ask(q.strip())\n  st.subheader("Answer"); st.write(answer)\n  a,b,c=st.columns(3); a.metric("Query route",intent.title()); b.metric("Search term",term[:30]); c.metric("Evidence records",len(records))\n  st.subheader("Retrieved graph evidence")\n  if records:\n   df=pd.DataFrame(records); st.dataframe(df,use_container_width=True,hide_index=True); st.download_button("Download evidence CSV",df.to_csv(index=False).encode(),file_name="cinegraph_evidence.csv",mime="text/csv")\n  else: st.info("No evidence returned. Confirm the sample graph is loaded.")\nst.divider(); st.caption("CineGraph · Neo4j + Cypher + Gemini")'
with open("/content/app.py","w",encoding="utf-8") as f: f.write(app_code)
print("Created /content/app.py")

In [ ]:
# Display the existing Streamlit app code
with open("/content/app.py", "r", encoding="utf-8") as file:
    print(file.read())

In [ ]:

%%writefile /content/app.py
import os
import re
import streamlit as st
import pandas as pd
from neo4j import GraphDatabase
from google import genai

st.set_page_config(
    page_title="CineGraph",
    page_icon="🎬",
    layout="wide"
)

# Connect to Neo4j and Gemini
driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(
        os.environ["NEO4J_USERNAME"],
        os.environ["NEO4J_PASSWORD"]
    )
)

gemini = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-2.5-flash"

# Queries match the properties in your current sample database.
QUERIES = {
    "movie": """
        MATCH (m:Movie)
        WHERE toLower(m.title) CONTAINS toLower($term)
        RETURN m.title AS title,
               m.released AS year,
               m.tagline AS tagline
        LIMIT 10
    """,

    "cast": """
        MATCH (p:Person)-[r:ACTED_IN]->(m:Movie)
        WHERE toLower(m.title) CONTAINS toLower($term)
        RETURN m.title AS movie,
               m.released AS year,
               p.name AS actor,
               r.roles AS roles
        LIMIT 30
    """,

    "actor": """
        MATCH (p:Person)-[r:ACTED_IN]->(m:Movie)
        WHERE toLower(p.name) CONTAINS toLower($term)
        RETURN p.name AS actor,
               m.title AS movie,
               m.released AS year,
               r.roles AS roles
        LIMIT 30
    """,

    "role": """
        MATCH (p:Person)-[r:ACTED_IN]->(m:Movie)
        WHERE any(role IN coalesce(r.roles, [])
                  WHERE toLower(role) CONTAINS toLower($term))
        RETURN p.name AS actor,
               m.title AS movie,
               r.roles AS roles
        LIMIT 20
    """,

    "director": """
        MATCH (p:Person)-[:DIRECTED]->(m:Movie)
        WHERE toLower(m.title) CONTAINS toLower($term)
        RETURN p.name AS director,
               m.title AS movie,
               m.released AS year
        LIMIT 20
    """,

    "recommend": """
        MATCH (m:Movie)
        WHERE toLower(m.title) CONTAINS toLower($term)
        MATCH (m)<-[:ACTED_IN]-(p:Person)-[:ACTED_IN]->(other:Movie)
        WHERE other <> m
        RETURN other.title AS title,
               other.released AS year,
               collect(DISTINCT p.name) AS shared_actors,
               count(DISTINCT p) AS connection_count
        ORDER BY connection_count DESC
        LIMIT 8
    """
}


def route(question):
    text = question.lower().strip()

    patterns = [
        ("recommend", [
            r"recommend movies like (.+)",
            r"recommend (.+)",
            r"movies like (.+)",
            r"films like (.+)",
            r"similar to (.+)"
        ]),
        ("role", [
            r"who voiced (.+)",
            r"who played (.+)",
            r"who portrayed (.+)"
        ]),
        ("director", [
            r"who directed (.+)",
            r"director of (.+)"
        ]),
        ("cast", [
            r"who acted in (.+)",
            r"cast of (.+)",
            r"actors in (.+)",
            r"who is in (.+)"
        ]),
        ("actor", [
            r"what movies did (.+?) act in",
            r"which movies did (.+?) act in",
            r"what films did (.+?) act in",
            r"filmography of (.+)"
        ]),
        ("movie", [
            r"tell me about (.+)",
            r"details (?:of|about) (.+)",
            r"information (?:on|about) (.+)"
        ])
    ]

    for intent, regexes in patterns:
        for pattern in regexes:
            match = re.search(pattern, text)
            if match:
                term = match.group(1).strip(" ?.!\"'")
                return intent, term

    return "movie", text.strip(" ?.!\"'")


def ask(question):
    intent, term = route(question)

    with driver.session() as session:
        records = [
            record.data()
            for record in session.run(
                QUERIES[intent],
                {"term": term}
            )
        ]

    if not records:
        return (
            "No matching graph evidence was found. "
            "Try a movie, actor, director, or character present in your database.",
            intent,
            term,
            records
        )

    prompt = (
        "Answer the question using ONLY the Neo4j evidence below. "
        "Do not invent facts. If the evidence is incomplete, say so. "
        "For recommendations, explain that shared actors are the basis "
        "for the recommendation.\n\n"
        f"Question: {question}\n"
        f"Evidence: {records}"
    )

    try:
        response = gemini.models.generate_content(
            model=MODEL,
            contents=prompt
        )
        answer = (response.text or "").strip()

        if not answer:
            answer = "Gemini returned an empty answer. See the evidence below."

    except Exception as error:
        answer = (
            f"Gemini could not generate an answer ({type(error).__name__}). "
            "The retrieved graph evidence is displayed below."
        )

    return answer, intent, term, records


# App design
st.title("🎬 CineGraph")
st.subheader("Graph Retrieval · Evidence First")
st.write(
    "Explore the connections between movies, actors, directors, "
    "and characters using your Neo4j graph."
)

st.sidebar.header("Try a question")

examples = [
    "Tell me about Toy Story",
    "Who acted in Toy Story?",
    "What movies did Tom Hanks act in?",
    "Who voiced Woody?",
    "Who directed The Matrix?",
    "Recommend movies like Heat"
]

selected = st.sidebar.selectbox(
    "Choose an example",
    ["Choose an example..."] + examples
)

st.sidebar.caption("Data source: your connected Neo4j graph")

question = st.text_input(
    "Ask CineGraph",
    value="" if selected == "Choose an example..." else selected,
    placeholder="For example: Who voiced Woody?"
)

if st.button("Find answer", type="primary", use_container_width=True):
    if not question.strip():
        st.warning("Please enter a question.")
    else:
        try:
            with st.spinner("Searching Neo4j and preparing your answer..."):
                answer, intent, term, records = ask(question.strip())

            st.header("Answer")
            st.write(answer)

            col1, col2, col3 = st.columns(3)
            col1.metric("Query type", intent.title())
            col2.metric("Search term", term)
            col3.metric("Evidence records", len(records))

            st.header("Retrieved graph evidence")

            if records:
                dataframe = pd.DataFrame(records)
                st.dataframe(
                    dataframe,
                    use_container_width=True,
                    hide_index=True
                )

                st.download_button(
                    "Download evidence as CSV",
                    dataframe.to_csv(index=False).encode("utf-8"),
                    file_name="cinegraph_evidence.csv",
                    mime="text/csv"
                )
            else:
                st.info(
                    "No matching records were found. "
                    "Try a different movie or person."
                )

        except Exception as error:
            st.error(f"Application error: {type(error).__name__}: {error}")

st.divider()
st.caption("CineGraph | Neo4j + Cypher + Gemini")


In [ ]:
import subprocess
import time
import os

subprocess.run(
    ["bash", "-lc", "pkill -f '[s]treamlit run /content/app.py' || true"],
    capture_output=True
)

time.sleep(2)

os.system(
    "streamlit run /content/app.py --server.port 8501 "
    "--server.address 0.0.0.0 > /content/streamlit.log 2>&1 &"
)

print("Streamlit restart command sent.")

## 10. Launch the web frontend from Colab

In [ ]:

import subprocess
import time

subprocess.run(
    ["bash", "-lc", "pkill -f '[s]treamlit run /content/app.py' || true"],
    capture_output=True
)

time.sleep(2)

with open("/content/streamlit.log", "w") as log_file:
    process = subprocess.Popen(
        [
            "streamlit", "run", "/content/app.py",
            "--server.port", "8501",
            "--server.address", "0.0.0.0"
        ],
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

time.sleep(4)

print("Streamlit running:", process.poll() is None)
print("\nRecent logs:")

with open("/content/streamlit.log", "r") as log_file:
    print(log_file.read()[-3000:])


In [ ]:
import os
os.environ["NEO4J_URI"]=NEO4J_URI
os.environ["NEO4J_USERNAME"]=NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"]=NEO4J_PASSWORD
os.environ["GEMINI_API_KEY"]=GEMINI_API_KEY
!streamlit run /content/app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

### Temporary demo URL
Run the next cell and open the URL it prints. A tunnel URL is temporary; do not publish it with credentials or leave a public demo running after submission.

In [ ]:

import subprocess
import time

# Stop old LocalTunnel processes
subprocess.run(
    ["bash", "-lc", "pkill -f '[l]ocaltunnel' || true"],
    capture_output=True
)

time.sleep(2)

# Start a fresh tunnel
tunnel = subprocess.Popen(
    ["npx", "--yes", "localtunnel", "--port", "8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

print("Waiting for a fresh URL...")

for _ in range(40):
    line = tunnel.stdout.readline()

    if line:
        print(line.strip())

        if "your url is:" in line.lower():
            break


In [ ]:

import requests

url = "http://127.0.0.1:8501"

try:
    response = requests.get(url, timeout=10)
    print("Streamlit status:", response.status_code)
    print("Page loaded:", response.status_code == 200)
    print("Response preview:", response.text[:300])
except Exception as e:
    print("Streamlit connection failed:", repr(e))


In [ ]:
!npm -s install -g localtunnel
!lt --port 8501

In [ ]:

import subprocess
import time

tunnel = subprocess.Popen(
    ["npx", "--yes", "localtunnel", "--port", "8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for _ in range(30):
    line = tunnel.stdout.readline()
    if line:
        print(line.strip())

    if line and "https://" in line:
        print("\nOpen the new URL above in your browser.")
        break


In [ ]:
import os

print("Gemini API key available:", bool(os.environ.get("GEMINI_API_KEY")))

try:
    from google import genai

    client = genai.Client(
        api_key=os.environ.get("GEMINI_API_KEY")
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="Reply with exactly: Gemini connection works."
    )

    print("Gemini response:", response.text)

except Exception as e:
    print("Error type:", type(e).__name__)
    print("Error details:", repr(e))

In [ ]:
import os
from google import genai

client = genai.Client(
    api_key=os.environ.get("GEMINI_API_KEY")
)

try:
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents="Reply with exactly: Gemini connection works."
    )

    print("SUCCESS!")
    print(response.text)

except Exception as e:
    print("Error type:", type(e).__name__)
    print("Error details:", repr(e))

In [ ]:

import os
import time
from google import genai

client = genai.Client(
    api_key=os.environ.get("GEMINI_API_KEY")
)

for attempt in range(1, 4):
    try:
        print(f"Attempt {attempt}...")

        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents="Reply with exactly: Gemini connection works."
        )

        print("SUCCESS!")
        print(response.text)
        break

    except Exception as e:
        print("Error:", repr(e))

        if attempt < 3:
            wait_seconds = attempt * 5
            print(f"Waiting {wait_seconds} seconds before retrying...")
            time.sleep(wait_seconds)
        else:
            print("Gemini is still unavailable after 3 attempts.")


In [ ]:

import os
from google import genai

client = genai.Client(
    api_key=os.environ.get("GEMINI_API_KEY")
)

try:
    print("Checking available Gemini models...\n")

    for model in client.models.list():
        name = getattr(model, "name", "") or ""

        if "flash" in name.lower():
            print(name)

except Exception as e:
    print("Error type:", type(e).__name__)
    print("Error details:", repr(e))


In [ ]:

import os
from google import genai

client = genai.Client(
    api_key=os.environ.get("GEMINI_API_KEY")
)

models_to_test = [
    "gemini-3.5-flash",
    "gemini-3.1-flash-lite",
    "gemini-flash-lite-latest"
]

for model_name in models_to_test:
    try:
        print(f"\nTesting {model_name}...")

        response = client.models.generate_content(
            model=model_name,
            contents="Reply with exactly: Connection works."
        )

        print("SUCCESS:", response.text)
        print("WORKING MODEL:", model_name)
        break

    except Exception as e:
        print("FAILED:", type(e).__name__, repr(e))


In [ ]:

from pathlib import Path

app_path = Path("/content/app.py")
code = app_path.read_text()

old_model = "gemini-2.5-flash"
new_model = "gemini-3.5-flash"

if old_model in code:
    code = code.replace(old_model, new_model)
    app_path.write_text(code)
    print("SUCCESS: CineGraph now uses gemini-3.5-flash.")
else:
    print("The old model name was not found.")
    print("Current Gemini model references:")

    for line in code.splitlines():
        if "gemini-" in line.lower():
            print(line)


In [ ]:

import subprocess
import time
import os

subprocess.run(
    ["bash", "-lc", "pkill -f '[s]treamlit run /content/app.py' || true"],
    capture_output=True
)

time.sleep(2)

with open("/content/streamlit.log", "w") as log_file:
    process = subprocess.Popen(
        [
            "streamlit", "run", "/content/app.py",
            "--server.port", "8501",
            "--server.address", "0.0.0.0"
        ],
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

time.sleep(3)

print("Streamlit process started.")
print("Process running:", process.poll() is None)
print("Log file: /content/streamlit.log")


## 11. Inspect graph edges for report analysis

### 15. Create a Robust Flask Backend & HTML/CSS Frontend

In [3]:
%%writefile /content/app.py
import os
import re
import pandas as pd
from flask import Flask, request, render_template_string, jsonify
from neo4j import GraphDatabase
from google import genai

app = Flask(__name__)

# Securely load credentials from environment variables
NEO4J_URI = os.environ.get("NEO4J_URI")
NEO4J_USERNAME = os.environ.get("NEO4J_USERNAME")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

# Configure drivers
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
gemini = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-3.5-flash"

QUERIES = {
    "movie": """
        MATCH (m:Movie)
        WHERE toLower(m.title) CONTAINS toLower($term)
        RETURN m.title AS title,
               m.released AS year,
               m.tagline AS tagline
        LIMIT 10
    """,

    "cast": """
        MATCH (p:Person)-[r:ACTED_IN]->(m:Movie)
        WHERE toLower(m.title) CONTAINS toLower($term)
        RETURN m.title AS movie,
               m.released AS year,
               p.name AS actor,
               r.roles AS roles
        LIMIT 30
    """,

    "actor": """
        MATCH (p:Person)-[r:ACTED_IN]->(m:Movie)
        WHERE toLower(p.name) CONTAINS toLower($term)
        RETURN p.name AS actor,
               m.title AS movie,
               m.released AS year,
               r.roles AS roles
        LIMIT 30
    """,

    "role": """
        MATCH (p:Person)-[r:ACTED_IN]->(m:Movie)
        WHERE any(role IN coalesce(r.roles, [])
                  WHERE toLower(role) CONTAINS toLower($term))
        RETURN p.name AS actor,
               m.title AS movie,
               r.roles AS roles
        LIMIT 20
    """,

    "director": """
        MATCH (p:Person)-[:DIRECTED]->(m:Movie)
        WHERE toLower(m.title) CONTAINS toLower($term)
        RETURN p.name AS director,
               m.title AS movie,
               m.released AS year
        LIMIT 20
    """,

    "recommend": """
        MATCH (m:Movie)
        WHERE toLower(m.title) CONTAINS toLower($term)
        MATCH (m)<-[:ACTED_IN]-(p:Person)-[:ACTED_IN]->(other:Movie)
        WHERE other <> m
        RETURN other.title AS title,
               other.released AS year,
               collect(DISTINCT p.name) AS shared_actors,
               count(DISTINCT p) AS connection_count
        ORDER BY connection_count DESC
        LIMIT 8
    """
}

def route(question):
    text = question.lower().strip()
    patterns = [
        ("recommend", [
            r"recommend movies like (.+)",
            r"recommend (.+)",
            r"movies like (.+)",
            r"films like (.+)",
            r"similar to (.+)"
        ]),
        ("role", [
            r"who voiced (.+)",
            r"who played (.+)",
            r"who portrayed (.+)"
        ]),
        ("director", [
            r"who directed (.+)",
            r"director of (.+)"
        ]),
        ("cast", [
            r"who acted in (.+)",
            r"cast of (.+)",
            r"actors in (.+)",
            r"who is in (.+)"
        ]),
        ("actor", [
            r"what movies did (.+?) act in",
            r"which movies did (.+?) act in",
            r"what films did (.+?) act in",
            r"filmography of (.+)"
        ]),
        ("movie", [
            r"tell me about (.+)",
            r"details (?:of|about) (.+)",
            r"information (?:on|about) (.+)"
        ])
    ]
    for intent, regexes in patterns:
        for pattern in regexes:
            match = re.search(pattern, text)
            if match:
                term = match.group(1).strip(" ?.!\"'")
                return intent, term
    return "movie", text.strip(" ?.!\"'")

def ask_cinegraph(question):
    intent, term = route(question)
    try:
        with driver.session() as session:
            records = [record.data() for record in session.run(QUERIES[intent], {"term": term})]
    except Exception as e:
        return f"Neo4j query failed: {str(e)}", intent, term, []

    if not records:
        return ("No matching graph evidence was found. Try a different query.", intent, term, [])

    prompt = (
        "Answer the question using ONLY the Neo4j evidence below. "
        "Do not invent facts. If the evidence is incomplete, say so. "
        "For recommendations, explain that shared actors are the basis "
        "for the recommendation.\n\n"
        f"Question: {question}\n"
        f"Evidence: {records}"
    )

    try:
        response = gemini.models.generate_content(model=MODEL, contents=prompt)
        answer = (response.text or "").strip()
        if not answer:
            answer = "Gemini returned an empty response."
    except Exception as e:
        answer = f"Gemini unavailable ({type(e).__name__}); inspect retrieved evidence below."

    return answer, intent, term, records

HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>🎬 CineGraph</title>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <style>
        body {
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
            background: linear-gradient(135deg, #111827, #0f172a);
            color: #f1f5f9;
            margin: 0;
            padding: 20px;
        }
        .container {
            max-width: 1000px;
            margin: 0 auto;
        }
        .header {
            background: linear-gradient(120deg, #1e293b, #312e81);
            padding: 30px;
            border-radius: 12px;
            box-shadow: 0 4px 15px rgba(0,0,0,0.5);
            margin-bottom: 25px;
            text-align: center;
        }
        h1 { margin: 0; font-size: 2.5rem; color: #38bdf8; }
        .subtitle { color: #94a3b8; font-size: 1.1rem; margin-top: 5px; }
        .grid {
            display: grid;
            grid-template-columns: 1fr 3fr;
            gap: 20px;
        }
        .sidebar {
            background: #1e293b;
            padding: 20px;
            border-radius: 10px;
            height: fit-content;
        }
        .sidebar h3 { margin-top: 0; color: #38bdf8; }
        .suggested-btn {
            display: block;
            width: 100%;
            text-align: left;
            background: #334155;
            color: #f1f5f9;
            border: none;
            padding: 10px 12px;
            margin-bottom: 8px;
            border-radius: 6px;
            cursor: pointer;
            font-size: 0.9rem;
            transition: background 0.2s;
        }
        .suggested-btn:hover { background: #475569; }
        .main-panel {
            background: #1e293b;
            padding: 25px;
            border-radius: 10px;
        }
        .input-group {
            display: flex;
            gap: 10px;
            margin-bottom: 20px;
        }
        input[type="text"] {
            flex: 1;
            padding: 12px;
            font-size: 1rem;
            border-radius: 6px;
            border: 1px solid #475569;
            background: #0f172a;
            color: #f1f5f9;
        }
        button[type="submit"] {
            background: #0284c7;
            color: white;
            border: none;
            padding: 12px 24px;
            font-weight: bold;
            border-radius: 6px;
            cursor: pointer;
            transition: background 0.2s;
        }
        button[type="submit"]:hover { background: #0369a1; }
        .answer-box {
            background: #0f172a;
            padding: 20px;
            border-radius: 8px;
            border-left: 4px solid #38bdf8;
            margin-top: 20px;
            line-height: 1.6;
        }
        .metrics {
            display: flex;
            gap: 15px;
            margin: 20px 0;
        }
        .metric-card {
            background: #334155;
            padding: 10px 15px;
            border-radius: 6px;
            flex: 1;
            text-align: center;
        }
        .metric-title { font-size: 0.8rem; color: #94a3b8; text-transform: uppercase; }
        .metric-value { font-size: 1.2rem; font-weight: bold; margin-top: 4px; color: #38bdf8; }
        table {
            width: 100%;
            border-collapse: collapse;
            margin-top: 15px;
            background: #0f172a;
            border-radius: 6px;
            overflow: hidden;
        }
        th, td {
            padding: 12px;
            text-align: left;
            border-bottom: 1px solid #334155;
        }
        th { background: #334155; color: #38bdf8; }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>🎬 CineGraph</h1>
            <div class="subtitle">Graph Retrieval &bull; Evidence First &bull; Flask Engine</div>
        </div>

        <div class="grid">
            <div class="sidebar">
                <h3>Try a Suggestion</h3>
                <button class="suggested-btn" onclick="setQuestion('Tell me about Toy Story')">Tell me about Toy Story</button>
                <button class="suggested-btn" onclick="setQuestion('Who acted in Toy Story?')">Who acted in Toy Story?</button>
                <button class="suggested-btn" onclick="setQuestion('What movies did Tom Hanks act in?')">What movies did Tom Hanks act in?</button>
                <button class="suggested-btn" onclick="setQuestion('Who voiced Woody?')">Who voiced Woody?</button>
                <button class="suggested-btn" onclick="setQuestion('Who directed The Matrix?')">Who directed The Matrix?</button>
                <button class="suggested-btn" onclick="setQuestion('Recommend movies like Heat')">Recommend movies like Heat</button>
            </div>

            <div class="main-panel">
                <form method="POST" action="/">
                    <div class="input-group">
                        <input type="text" name="question" id="qInput" placeholder="For example: Who voiced Woody?" value="{{ question }}" required>
                        <button type="submit">Find Answer</button>
                    </div>
                </form>

                {% if answer %}
                    <div class="metrics">
                        <div class="metric-card">
                            <div class="metric-title">Query Type</div>
                            <div class="metric-value">{{ intent|upper }}</div>
                        </div>
                        <div class="metric-card">
                            <div class="metric-title">Search Term</div>
                            <div class="metric-value">{{ term }}</div>
                        </div>
                        <div class="metric-card">
                            <div class="metric-title">Records Found</div>
                            <div class="metric-value">{{ records|length }}</div>
                        </div>
                    </div>

                    <h3>Answer</h3>
                    <div class="answer-box">
                        {{ answer }}
                    </div>

                    {% if records %}
                        <h3>Retrieved Graph Evidence</h3>
                        <div style="overflow-x:auto;">
                            <table>
                                <thead>
                                    <tr>
                                        {% for key in records[0].keys() %}
                                            <th>{{ key|capitalize }}</th>
                                        {% endfor %}
                                    </tr>
                                </thead>
                                <tbody>
                                    {% for row in records %}
                                        <tr>
                                            {% for val in row.values() %}
                                                <td>{{ val }}</td>
                                            {% endfor %}
                                        </tr>
                                    {% endfor %}
                                </tbody>
                            </table>
                        </div>
                    {% endif %}
                {% endif %}
            </div>
        </div>
    </div>

    <script>
        function setQuestion(text) {
            document.getElementById('qInput').value = text;
        }
    </script>
</body>
</html>
"""

@app.route("/", methods=["GET", "POST"])
def home():
    question = ""
    answer = ""
    intent = ""
    term = ""
    records = []
    if request.method == "POST":
        question = request.form.get("question", "").strip()
        if question:
            answer, intent, term, records = ask_cinegraph(question)
    return render_template_string(HTML_TEMPLATE, question=question, answer=answer, intent=intent, term=term, records=records)

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8501)

Writing /content/app.py


### 16. Start the Flask Server and LocalTunnel Link

In [ ]:
import subprocess
import time
import os

# Terminate any old streamlit/flask instances running on port 8501
print("Stopping existing servers...")
subprocess.run(["bash", "-lc", "pkill -f '[a]pp.py' || true"], capture_output=True)
time.sleep(2)

# Verify dependencies for Flask are present
%pip install -q flask

# Export environmental variables securely to the background task
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# Spin up Flask in the background on port 8501
print("Launching Flask Server...")
with open("/content/flask.log", "w") as log_file:
    process = subprocess.Popen(
        ["python", "/content/app.py"],
        stdout=log_file,
        stderr=subprocess.STDOUT
    )
time.sleep(3)

print("Flask running in background:", process.poll() is None)

In [ ]:
# Re-start a clean LocalTunnel instance pointing to Flask (port 8501)
subprocess.run(["bash", "-lc", "pkill -f '[l]ocaltunnel' || true"], capture_output=True)
time.sleep(2)

tunnel = subprocess.Popen(
    ["npx", "--yes", "localtunnel", "--port", "8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Acquiring Tunnel URL...")
for _ in range(30):
    line = tunnel.stdout.readline()
    if "your url is:" in line.lower() or "https://" in line:
        print("\n🚀 Click the link below to open your running Flask application:")
        print(line.strip())
        break

In [ ]:
edges=run_cypher("MATCH (p:Person)-[r:ACTED_IN]->(m:Movie) RETURN p.name AS person,type(r) AS relationship,m.title AS movie,coalesce(r.roles,r.role) AS role LIMIT 25")
display(pd.DataFrame(edges))

In [ ]:
import pandas as pd
# Query all Movie titles to see what exactly is inside your database
movies_in_db = run_cypher("MATCH (m:Movie) RETURN m.title AS title, m.released AS released, keys(m) AS properties LIMIT 100")
display(pd.DataFrame(movies_in_db))

In [ ]:
# Show the relationship counts per movie to highlight why Toy Story / Heat have no evidence
movie_relationships = run_cypher("""
    MATCH (m:Movie)
    OPTIONAL MATCH (m)-[r]-()
    RETURN m.title AS Movie, count(r) AS RelationshipCount
""")
import pandas as pd
display(pd.DataFrame(movie_relationships))

## 12. Evaluation checklist
Record actual results after running: (1) database connects; (2) schema inspected; (3) movie lookup returns evidence; (4) cast and actor lookups work; (5) role query tested; (6) recommendations explained as shared-actor results; (7) UI shows answer and evidence; (8) unknown titles return no-evidence response; (9) errors handled. Do not claim tests pass until verified.

## 13. Report outline
Follow the uploaded DOCX template's section order and replace example content with your implementation: Aim/Objectives; Expected Outcomes; Tools; Dataset; Architecture; Backend; Frontend; Sample Inputs/Outputs; Evaluation; Challenges; Conclusion/Future Work; Repository; References. Include actual screenshots and results; do not invent precision/recall/F1 scores.

References: Neo4j example data https://neo4j.com/developer/example-data/ · Cypher https://neo4j.com/docs/cypher-manual/current/ · Gemini https://ai.google.dev/gemini-api/docs · Streamlit https://docs.streamlit.io/ · Lewis et al. (2020), RAG paper https://arxiv.org/abs/2005.11401

## 14. Cleanup

In [ ]:
driver.close()
print("Neo4j driver closed.")

In [ ]:
%%writefile requirements.txt
Flask
neo4j
google-genai
pandas

In [ ]:
%%writefile .gitignore
.env
.env.*
!.env.example
*.pem
secrets.json
__pycache__/
*.pyc
.ipynb_checkpoints/

In [5]:
from google.colab import files
files.download("/content/app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>